# **data**

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tongpython/cat-and-dog")

print("Path to dataset files:", path)

100%|██████████| 218M/218M [00:16<00:00, 13.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1


In [4]:
import os

path = "/root/.cache/kagglehub/datasets/tongpython/cat-and-dog/versions/1"

print(os.listdir(path))

['test_set', 'training_set']


In [7]:

train_dir = path + "/training_set/training_set"
test_dir  = path + "/test_set/test_set"

# **image data generator**

In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training data (with augmentation)
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

# Testing data (no augmentation)
test_gen = ImageDataGenerator(rescale=1./255)

In [10]:
train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

Found 8005 images belonging to 2 classes.


In [11]:
test_data = test_gen.flow_from_directory(
    test_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

Found 2023 images belonging to 2 classes.


## **MODEL**

In [12]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# **Training**

In [13]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [14]:
model.fit(train_data, epochs=5)

Epoch 1/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 61s 225ms/step - accuracy: 0.5799 - loss: 0.7019
Epoch 2/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 56s 223ms/step - accuracy: 0.6648 - loss: 0.6184
Epoch 3/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 54s 216ms/step - accuracy: 0.6937 - loss: 0.5829
Epoch 4/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 56s 223ms/step - accuracy: 0.7167 - loss: 0.5550
Epoch 5/5
251/251 ━━━━━━━━━━━━━━━━━━━━ 53s 211ms/step - accuracy: 0.7270 - loss: 0.5405


# **Testing**

In [15]:
loss, accuracy = model.evaluate(test_data)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy*100:.2f}%")

64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 58ms/step - accuracy: 0.7662 - loss: 0.5050
Test Loss: 0.5050
Test Accuracy: 76.62%
